In [ ]:
# Supplementary Table S2
# Per-glycopeptiform summary of the integrative analysis previously shown as the
# Figure 3 upset / bar panels (removed from the main figure).
# One row per glycopeptiform (gpID). Row set = glycopeptiforms significant in the
# unadjusted glyco-TWAS (GlycoWAS) reference set UNION glycopeptiforms colocalized with AD.
# Columns cover: (i) glycopeptiform annotation, (ii) GlycoWAS (unadjusted),
# (iii) PWAS (inherited from the parent protein), (iv) colocalization with AD / pQTL
# with effect directionality at the colocalization lead variant.
#
# Sources are kept internally consistent: GlycoWAS / PWAS from the TWAS result table;
# AD colocalization membership AND direction both taken from the maf05 colocalization bed,
# which stores the per-outcome effect (coef / zscore) at each colocalization signal (CoS).

suppressMessages({library(data.table); library(tidyverse); library(openxlsx)})

twas_path  <- "Work/gpQTL/data_output/twas/glyco_v202601.twas_res.no_ashinf.csv.gz"
bed_path   <- "data_output/colocalization/maf05/glycoQTL_maf05/pQTL_glycoQTL_unadjusted_AD_pv1e4.bed"
anno_path  <- "Work/gpQTL/Annotation_with_gp_coordinates_clean.txt"
out_xlsx   <- "TableS2.xlsx"

stopifnot(file.exists(path.expand(twas_path)), file.exists(bed_path), file.exists(path.expand(anno_path)))

twas <- fread(twas_path)
anno <- fread(anno_path, sep = "\t")
bed  <- fread(bed_path)

# TMEM106B: the older maf05 bed rows are replaced by the v202609 rerun (section B2 below).
tmem_id <- "ENSG00000106460"
bed <- bed[region_ID != tmem_id]

# ---------------------------------------------------------------------------
# (A) GlycoWAS (unadjusted) per glycopeptiform.
# Significance rule identical to the figure notebook: Bonferroni cutoff over imputable
# (molecular_id, context, gwas_study) tests; a context passes if >=50% of its methods are
# significant OR the selected method is significant. The representative p / z is the row
# matching the final_pval (best selected-method p, else best p).
# ---------------------------------------------------------------------------
was_summary <- function(tp) {
  d <- twas[type == tp]
  cutoff <- 0.05 / d[is_imputable == TRUE, uniqueN(paste(molecular_id, context, gwas_study))]
  s <- d[, .(
    n_methods        = .N,
    n_sig            = sum(twas_pval < cutoff, na.rm = TRUE),
    best_pval        = min(twas_pval, na.rm = TRUE),
    best_method_pval = suppressWarnings(min(twas_pval[is_selected_method], na.rm = TRUE)),
    best_sig         = any(twas_pval < cutoff & is_selected_method, na.rm = TRUE)
  ), by = .(molecular_id, context, gwas_study)]
  s[, pass := (n_sig / n_methods >= 0.5) | best_sig]
  s[, final_pval := fifelse(n_sig / n_methods >= 0.5, best_pval, best_method_pval)]
  s[, final_pval := fifelse(is.infinite(final_pval), best_pval, final_pval)]
  # attach the z of the representative row (signed direction of the WAS effect)
  z <- d[s, on = .(molecular_id, context, gwas_study)][twas_pval == final_pval]
  z <- z[, .SD[1], by = .(molecular_id, context, gwas_study)]
  list(cutoff = cutoff,
       tab = z[, .(molecular_id, context, gwas_study,
                   z = twas_z, pval = final_pval, sig = pass)])
}

glyco <- was_summary("unadjusted_gpQTL")
pwas  <- was_summary("pQTL")

# glycopeptiform-level GlycoWAS table
glyco_tab <- glyco$tab[, .(
  gpID            = str_extract(context, "gp_[0-9]+$"),
  ensembl_id      = molecular_id,
  glycowas_z      = z,
  glycowas_pval   = pval,
  glycowas_sig    = sig
)]

# protein-level PWAS table (one row per ensembl_id; context is DLPFC_Bennett_pQTL)
pwas_tab <- pwas$tab[, .(
  ensembl_id   = molecular_id,
  pwas_z       = z,
  pwas_pval    = pval,
  pwas_sig     = sig
)]

# ---------------------------------------------------------------------------
# (B) AD colocalization + directionality from the maf05 bed.
# event_ID / coef / zscore are parallel ; lead variant per CoS = max vcp.
# A CoS is AD-colocalized if its event set contains AD_Bellenguez.
# At the lead variant all outcome effects are coded to the SAME allele, so the sign
# relationship between the AD effect (z_AD) and the glycopeptiform effect (z_gp) tells
# whether the AD-risk-increasing allele raises or lowers glycopeptiform abundance,
# independent of which allele is labelled the effect allele.
# ---------------------------------------------------------------------------
ad_cos <- unique(bed[str_detect(event_ID, "AD_Bellenguez"), cos_ID])
# .I returns row positions in the original bed; index bed directly (lead = max-vcp variant per CoS)
lead <- bed[bed[cos_ID %in% ad_cos, .I[which.max(vcp)], by = cos_ID]$V1]

dir_dt <- lead[, {
  ev <- str_split(event_ID, "; *")[[1]]
  z  <- as.numeric(str_split(zscore, "; *")[[1]])
  b  <- as.numeric(str_split(coef,   "; *")[[1]])
  data.table(outcome = ev, z = z, coef = b)
}, by = .(cos_id = cos_ID, ensembl_id = region_ID, lead_variant = variant_ID, a1, a2)]

# per-CoS AD effect and per-CoS parent-protein pQTL effect
ad_z   <- dir_dt[outcome == "AD_Bellenguez", .(cos_id, z_AD = z)]
pqtl_z <- dir_dt[str_detect(outcome, "Bennett_pQTL_ENSG"),
                 .(cos_id, pqtl_ensembl = str_extract(outcome, "ENSG[0-9]+"), z_pQTL = z)]

# glycopeptiform rows within AD-colocalized CoS
gp_dir <- dir_dt[str_detect(outcome, "gpQTL_unadjusted_gp_")][
  , .(gpID = str_extract(outcome, "gp_[0-9]+$"), cos_id, ensembl_id, lead_variant, a1, a2, z_gp = z)]
gp_dir <- ad_z[gp_dir, on = "cos_id"]
gp_dir <- merge(gp_dir, pqtl_z, by.x = c("cos_id", "ensembl_id"),
                by.y = c("cos_id", "pqtl_ensembl"), all.x = TRUE)

# if a glycopeptiform is in >1 AD-coloc CoS, keep the CoS where its QTL signal is strongest
gp_dir[, n_ad_coloc_cos := uniqueN(cos_id), by = gpID]
gp_dir <- gp_dir[order(gpID, -abs(z_gp))][, .SD[1], by = gpID]

coloc_tab <- gp_dir[, .(
  gpID,
  ad_coloc          = TRUE,
  coloc_cos_id      = cos_id,
  coloc_lead_variant = lead_variant,
  z_gpQTL_at_lead   = round(z_gp, 3),
  z_AD_at_lead      = round(z_AD, 3),
  z_pQTL_at_lead    = round(z_pQTL, 3),
  # AD-risk allele vs glycopeptiform abundance
  ad_risk_vs_gp_abundance = fifelse(sign(z_AD) == sign(z_gp), "higher", "lower"),
  pqtl_coloc        = !is.na(z_pQTL),
  # does protein abundance move the same way as glycopeptiform abundance for this allele
  gp_pqtl_same_direction = fifelse(!is.na(z_pQTL), sign(z_gp) == sign(z_pQTL), NA),
  n_ad_coloc_cos
)]

# ---------------------------------------------------------------------------
# (B2) TMEM106B AD colocalization from the v202609 rerun (unadjusted glycoQTL + parent-protein
# pQTL + AD GWAS; colocboost object unadj_glyco_AD). Same rules as (B): AD-colocalized CoS only,
# lead variant = top_variable (max vcp) of the CoS, per-outcome z taken at that variant (all
# outcomes are coded to the same allele in data_info$z), direction from sign(z_AD) vs sign(z_gp).
# ---------------------------------------------------------------------------
tmem_cb_path <- "data_output/TMEM106B_rerun_202609/glycoQTL_tmem106b/glycoQTL_unadjusted_AD/colocboost/unadj_glyco_AD.chr7_ENSG00000106460.cb_xqtl_AD_Bellenguez.rds"
stopifnot(file.exists(tmem_cb_path))
tcb <- readRDS(tmem_cb_path)[[1]]
tcs <- as.data.table(tcb$cos_summary)[grepl("AD_Bellenguez", colocalized_outcomes)]
stopifnot(nrow(tcs) >= 1)
tmem_dir <- rbindlist(lapply(seq_len(nrow(tcs)), function(i) {
  outs <- str_split(tcs$colocalized_outcomes[i], "; *")[[1]]
  lead_v <- tcs$top_variable[i]; j <- match(lead_v, tcb$data_info$variables); stopifnot(!is.na(j))
  zz <- vapply(outs, function(o) tcb$data_info$z[[o]][j], numeric(1))
  gps <- outs[str_detect(outs, "gpQTL_unadjusted_gp_")]
  data.table(gpID = str_extract(gps, "gp_[0-9]+$"), cos_id = tcs$cos_id[i], lead_variant = lead_v,
             z_gp = unname(zz[gps]), z_AD = unname(zz["AD_Bellenguez"]),
             z_pQTL = if (any(str_detect(outs, "Bennett_pQTL_ENSG"))) unname(zz[str_detect(outs, "Bennett_pQTL_ENSG")][1]) else NA_real_)
}))
tmem_dir[, n_ad_coloc_cos := uniqueN(cos_id), by = gpID]
tmem_dir <- tmem_dir[order(gpID, -abs(z_gp))][, .SD[1], by = gpID]
tmem_tab <- tmem_dir[, .(
  gpID, ad_coloc = TRUE, coloc_cos_id = cos_id, coloc_lead_variant = lead_variant,
  z_gpQTL_at_lead = round(z_gp, 3), z_AD_at_lead = round(z_AD, 3), z_pQTL_at_lead = round(z_pQTL, 3),
  ad_risk_vs_gp_abundance = fifelse(sign(z_AD) == sign(z_gp), "higher", "lower"),
  pqtl_coloc = !is.na(z_pQTL), gp_pqtl_same_direction = fifelse(!is.na(z_pQTL), sign(z_gp) == sign(z_pQTL), NA),
  n_ad_coloc_cos
)]
coloc_tab <- rbind(coloc_tab, tmem_tab, use.names = TRUE)

# ---------------------------------------------------------------------------
# (C) Annotation per glycopeptiform.
# ---------------------------------------------------------------------------
anno_tab <- anno[, .(
  gpID, gene_name, ensembl_id,
  chr = `#chr`, start, end,
  GlycanClass, GlycanTypeSummary, GlycoSite, GlycanComposition, peptide_ID
)]

# ---------------------------------------------------------------------------
# (D) Assemble. Row set = GlycoWAS-significant (unadjusted) UNION AD-colocalized.
# ---------------------------------------------------------------------------
keep_gp <- union(glyco_tab[glycowas_sig == TRUE, gpID], coloc_tab[, gpID])

tab <- data.table(gpID = keep_gp) |>
  merge(anno_tab, by = "gpID", all.x = TRUE)
# carry ensembl_id from annotation; fall back to GlycoWAS ensembl_id when annotation missing
tab <- merge(tab, glyco_tab, by = c("gpID", "ensembl_id"), all.x = TRUE)
tab <- merge(tab, pwas_tab,  by = "ensembl_id", all.x = TRUE)
tab <- merge(tab, coloc_tab, by = "gpID", all.x = TRUE)

# fill logical/flag defaults
tab[, glycowas_tested := gpID %in% glyco_tab$gpID]
tab[, pwas_tested     := ensembl_id %in% pwas_tab$ensembl_id]
tab[is.na(glycowas_sig), glycowas_sig := FALSE]
tab[is.na(pwas_sig),     pwas_sig := FALSE]
tab[is.na(ad_coloc),     ad_coloc := FALSE]
tab[is.na(pqtl_coloc),   pqtl_coloc := FALSE]
tab[is.na(n_ad_coloc_cos), n_ad_coloc_cos := 0L]

# lean column set: drop redundant annotation (GlycanTypeSummary, peptide_ID), gene coordinates
# (chr / start / end), tested flags, raw per-outcome z, and CoS technical columns
# (direction is summarised by the two flags below)
keep_cols <- c(
  # (i) annotation
  "gene_name", "gpID", "ensembl_id",
  "GlycoSite", "GlycanComposition", "GlycanClass",
  # (ii) GlycoWAS (unadjusted)
  "glycowas_z", "glycowas_pval", "glycowas_sig",
  # (iii) PWAS
  "pwas_z", "pwas_pval", "pwas_sig",
  # (iv) colocalization + direction
  "ad_coloc", "pqtl_coloc", "coloc_lead_variant",
  "ad_risk_vs_gp_abundance", "gp_pqtl_same_direction"
)
tab <- tab[, ..keep_cols]
tab <- tab[order(gene_name, gpID)]   # alphabetical by gene, then glycopeptiform ID

# capture summary stats before renaming (columns are renamed below)
n_rows <- nrow(tab); n_gwsig <- sum(tab$glycowas_sig); n_adco <- sum(tab$ad_coloc)
n_prot <- uniqueN(tab$ensembl_id); n_pqco <- sum(tab$pqtl_coloc)
n_samedir <- sum(tab$gp_pqtl_same_direction, na.rm = TRUE)

# readable column names
setnames(tab, c(
  "Gene", "Glycopeptiform ID", "Ensembl gene ID",
  "Glycosite", "Glycan composition", "Glycan class",
  "GlycoWAS z", "GlycoWAS P", "GlycoWAS significant",
  "PWAS z", "PWAS P", "PWAS significant",
  "AD colocalized", "pQTL colocalized", "Colocalization lead variant",
  "AD-risk allele vs glycopeptiform abundance", "Glycopeptiform and protein same direction"
))

# ---------------------------------------------------------------------------
# (E) Column dictionary sheet.
# ---------------------------------------------------------------------------
dict <- data.table(
  column = colnames(tab),
  description = c(
    "Glycopeptiform identifier (gpID)",
    "Parent gene symbol",
    "Parent gene Ensembl ID",
    "Glycosylation site (UniProt_residue)", "Glycan composition", "Glycan class",
    "Unadjusted glyco-TWAS z (signed; representative method)",
    "Unadjusted glyco-TWAS p (final_pval; best selected-method p, else best p)",
    "Unadjusted glyco-TWAS significant (>=50% methods sig OR selected method sig, Bonferroni)",
    "PWAS z (signed; representative method; parent protein DLPFC_Bennett_pQTL)",
    "PWAS p (final_pval)",
    "PWAS significant (same rule as GlycoWAS)",
    "Glycopeptiform colocalized with AD (Bellenguez 2022; maf05 colocalization)",
    "Parent-protein pQTL also colocalized in the same AD colocalization signal",
    "Lead variant (max vcp) of the AD colocalization signal",
    "AD-risk-increasing allele associated with higher/lower glycopeptiform abundance",
    "TRUE if glycopeptiform and protein abundance move the same direction for that allele (NA if no pQTL)"
  )
)

wb <- createWorkbook()
addWorksheet(wb, "S4_glycopeptiform_summary")
addWorksheet(wb, "column_dictionary")
writeData(wb, "S4_glycopeptiform_summary", tab)
writeData(wb, "column_dictionary", dict)
freezePane(wb, "S4_glycopeptiform_summary", firstRow = TRUE)
setColWidths(wb, "S4_glycopeptiform_summary", cols = 1:ncol(tab), widths = "auto")
setColWidths(wb, "column_dictionary", cols = 1:2, widths = c(28, 90))
saveWorkbook(wb, out_xlsx, overwrite = TRUE)
# Footer note for the combined workbook (Build_Supplementary_Tables_Excel.ipynb moves a last row that starts with "Note:" below the table).
# Counts and cutoffs come from the objects above, so the note cannot drift from the table.
fmt_sci <- function(x) { e <- floor(log10(x)); sprintf("%.2f × 10⁻%s", x / 10^e, chartr("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹", as.character(abs(e)))) }
note <- sprintf(paste0(
  "Note: Rows are glycopeptiforms (gpID) that are significant in the unadjusted GlycoWAS or colocalized with AD (%d glycopeptiforms from %d proteins; %d GlycoWAS significant, %d AD colocalized)."),
  n_rows, n_prot, n_gwsig, n_adco)
fwrite(tab, "TableS2.csv")
cat(sprintf('"%s"%s\n', gsub('"', '""', note), strrep(",", ncol(tab) - 1L)), file = "TableS2.csv", append = TRUE)

cat(sprintf("GlycoWAS unadjusted cutoff: %.3e | PWAS cutoff: %.3e\n", glyco$cutoff, pwas$cutoff))
cat(sprintf("Rows (glycopeptiforms): %d  [GlycoWAS-sig: %d ; AD-coloc: %d]\n", n_rows, n_gwsig, n_adco))
cat(sprintf("Proteins (unique ensembl_id): %d\n", n_prot))
cat(sprintf("AD-coloc with pQTL coloc: %d | same-direction gp~protein: %d\n", n_pqco, n_samedir))
cat("Wrote", out_xlsx, "and TableS2.csv\n")


## QC: sole-modification Trp C-mannosylation exclusion check

54 glycopeptiforms whose only modification is Trp C-mannosylation are mislabeled as `glycation` throughout the annotation (see Figure1_data_preparation.ipynb and chat discussion). Checked whether any of them appear in the 'Glycopeptiform ID' column of TableS4 -- none do, so no change to this table is needed.

In [ ]:
qc_remove_gpids <- readLines('../2_main_figures/Figure1/data/qc_remove_gpids.txt')
w_sole_exclude_gpids <- readLines('../2_main_figures/Figure1/data/w_sole_exclude_gpids.txt')
stopifnot(length(w_sole_exclude_gpids) == 54)
excluded_gpids <- c(qc_remove_gpids, w_sole_exclude_gpids)
tbl <- read.csv('TableS2.csv', check.names = FALSE)
hits <- intersect(unique(tbl[['Glycopeptiform ID']]), excluded_gpids)
stopifnot(length(hits) == 0)
cat('No affected gpIDs found in', nrow(tbl), 'rows of TableS2.csv
')